# Module 7: Deep RL and Representation Learning

**Spinning Up in Active Inference** | Part 2: The RL Track

---

## Historical Context

In February 2015, a paper in *Nature* shook the AI world. Mnih et al. showed that a single neural network, given only raw pixel inputs and game score, could learn to play 49 Atari games at human level. **Deep Q-Networks** (DQN) married deep learning with Q-learning, and deep RL was born.

But DQN was more than a engineering triumph. It revealed something fundamental: **representation learning**. The network's hidden layers automatically discovered features—edges, objects, spatial relationships—that were useful for decision-making. The agent learned *what to pay attention to*.

This connects directly to Active Inference. Where deep RL learns representations *implicitly* through reward gradients, AIF defines a **generative model** $P(o|s)$ that *explicitly* specifies the mapping from hidden states to observations. Deep Active Inference (Module 12) will bridge these by parameterizing the generative model with neural networks.

### What You'll Learn

1. Why tabular methods fail at scale: the curse of dimensionality
2. Function approximation: $V(s; \theta) \approx V^*(s)$
3. DQN: experience replay + target networks
4. PPO and SAC: the modern workhorses
5. What representations do deep RL agents learn?

> **Note**: This module focuses on *understanding* deep RL concepts and their connection to AIF. For hands-on deep RL training, see the excellent [neuro-nav deep RL tutorial](https://github.com/awjuliani/neuro-nav/blob/main/tutorials/deeprl_tutorial.ipynb).

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import sys
sys.path.insert(0, '..')

from neuronav.agents.td_agents import TDQ, TDSR
from neuronav.envs.grid_env import GridEnv, GridSize, GridObservation
from neuronav.envs.grid_templates import GridTemplate

from utils.plotting import (plot_value_heatmap, plot_matrix_heatmap,
                            plot_learning_curve, dual_perspective_box, COLORS)

np.random.seed(42)
print("Module 7: Deep RL and Representation Learning")
print("="*50)

## 7.1 The Curse of Dimensionality

All the methods we've studied so far are **tabular**: they maintain a table with one entry per state (or state-action pair). This works beautifully for small environments, but breaks catastrophically at scale.

Consider a 10x10 grid: 100 states, 4 actions = 400 entries. Manageable.

But for realistic problems:
- Atari: 210 x 160 pixels x 3 colors x 256 levels = $256^{100800}$ possible frames
- Go: $3^{361} \approx 10^{172}$ board positions
- Robotics: continuous states (joint angles, velocities) = infinite states

We need **function approximation**: learn a compact function that generalizes across similar states.

In [ ]:
# Demonstrate the scaling problem
grid_sizes = [5, 10, 20, 50, 100, 200]
state_spaces = [n**2 for n in grid_sizes]
q_table_sizes = [n**2 * 4 for n in grid_sizes]  # 4 actions
memory_mb = [size * 8 / 1e6 for size in q_table_sizes]  # float64 = 8 bytes

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# State space growth
axes[0].semilogy(grid_sizes, state_spaces, 'o-', color=COLORS['rl'],
                 linewidth=2, markersize=8)
axes[0].fill_between(grid_sizes, state_spaces, alpha=0.1, color=COLORS['rl'])
axes[0].set_xlabel("Grid Side Length")
axes[0].set_ylabel("Number of States (log scale)")
axes[0].set_title("State Space Grows Quadratically")
axes[0].grid(True, alpha=0.3)

# Add annotations
for i, (gs, ss) in enumerate(zip(grid_sizes, state_spaces)):
    if i % 2 == 0:
        axes[0].annotate(f"{gs}x{gs}\n= {ss:,} states",
                         (gs, ss), textcoords="offset points",
                         xytext=(10, 5), fontsize=8)

# Real-world comparison
problems = ['5x5\nGrid', '10x10\nGrid', '100x100\nGrid', 'Atari\n(pixels)',
            'Go\n(board)', 'Robot\n(joints)']
sizes = [25, 100, 10000, 1e8, 1e172, float('inf')]
bar_heights = [np.log10(max(s, 1)) if np.isfinite(s) else 200 for s in sizes]
bar_colors = [COLORS['success'], COLORS['success'], COLORS['highlight'],
              COLORS['danger'], COLORS['danger'], COLORS['danger']]

axes[1].bar(range(len(problems)), bar_heights, color=bar_colors, alpha=0.8)
axes[1].set_xticks(range(len(problems)))
axes[1].set_xticklabels(problems, fontsize=9)
axes[1].set_ylabel("log10(State Space Size)")
axes[1].set_title("Tabular Methods Can't Scale")
axes[1].axhline(y=4, color='red', linestyle='--', alpha=0.5,
                label='Tabular feasibility limit (~10^4)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle("The Curse of Dimensionality", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTabular RL is feasible up to ~10,000 states.")
print("Beyond that, we need function approximation.")

## 7.2 Function Approximation: The Basic Idea

Instead of storing $V(s)$ for every state, we approximate it with a parameterized function:

$$V(s; \theta) \approx V^*(s)$$

The simplest case is **linear function approximation**:

$$V(s; \theta) = \theta^\top \phi(s)$$

where $\phi(s)$ is a **feature vector** that captures relevant properties of state $s$.

The key question: what features to use? In deep RL, the neural network learns the features automatically. But let's start with hand-crafted features to build intuition.

In [ ]:
# Linear function approximation: learn V(s) with hand-crafted features

class LinearVFA:
    """Linear value function approximation with hand-crafted features."""

    def __init__(self, grid_size, goal_pos, lr=0.01, gamma=0.99):
        self.grid_size = grid_size
        self.goal_pos = goal_pos
        self.lr = lr
        self.gamma = gamma
        # 5 features: x, y, dist_to_goal, x^2, y^2
        self.n_features = 5
        self.theta = np.zeros(self.n_features)

    def features(self, state):
        """Extract features from a state index."""
        row = state // self.grid_size
        col = state % self.grid_size
        # Normalize to [0, 1]
        x = col / self.grid_size
        y = row / self.grid_size
        # Manhattan distance to goal (normalized)
        gx = self.goal_pos[1] / self.grid_size
        gy = self.goal_pos[0] / self.grid_size
        dist = abs(x - gx) + abs(y - gy)
        return np.array([x, y, dist, x**2, y**2])

    def value(self, state):
        return self.theta @ self.features(state)

    def update_td(self, s, r, s_next, done):
        """TD(0) update with linear function approximation."""
        phi_s = self.features(s)
        v_s = self.theta @ phi_s
        v_next = 0 if done else self.theta @ self.features(s_next)
        td_error = r + self.gamma * v_next - v_s
        # Gradient of V(s;theta) = phi(s) for linear case
        self.theta += self.lr * td_error * phi_s
        return td_error


# Train on a simple grid
grid_size_demo = 10
goal_pos = (9, 9)  # Bottom-right corner

env = GridEnv(obs_type=GridObservation.index, size=GridSize.small)
state_size = env.grid_size ** 2
action_size = env.action_space.n
actual_grid_size = env.grid_size

# Use actual grid size for features
vfa = LinearVFA(actual_grid_size, goal_pos=(actual_grid_size-1, actual_grid_size-1),
                lr=0.01, gamma=0.99)

# Also train a tabular agent for comparison
tdq_tab = TDQ(state_size, action_size, lr=0.1, gamma=0.99,
              poltype="softmax", beta=5.0)

td_errors_vfa = []
td_errors_tab = []

for ep in range(300):
    obs = env.reset()
    ep_tde_vfa = []
    for step in range(300):
        # Random policy for VFA (just learning value)
        action = np.random.randint(action_size)
        next_obs, reward, done, info = env.step(action)

        # Update VFA
        tde = vfa.update_td(obs, reward, next_obs, done)
        ep_tde_vfa.append(abs(tde))

        # Update tabular
        tdq_tab.update((obs, action, next_obs, reward, done))

        obs = next_obs
        if done:
            break

    td_errors_vfa.append(np.mean(ep_tde_vfa))

print(f"Learned weights: {vfa.theta.round(3)}")
print(f"Feature names: [x, y, dist_to_goal, x^2, y^2]")
print(f"\nKey: distance feature (weight={vfa.theta[2]:.3f}) dominates")
print("  -> Value decreases with distance to goal (as expected)")

In [ ]:
# Compare tabular vs linear VFA representations
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Tabular V(s)
# Q shape is (action_size, state_size) — max over axis=0 gives V(s)
v_tab = np.max(tdq_tab.Q, axis=0)
if actual_grid_size**2 == state_size:
    plot_value_heatmap(v_tab, actual_grid_size,
                       title="Tabular V(s)\n(one value per state)",
                       ax=axes[0], show_values=False)

# Linear VFA V(s)
v_vfa = np.array([vfa.value(s) for s in range(state_size)])
if actual_grid_size**2 == state_size:
    plot_value_heatmap(v_vfa, actual_grid_size,
                       title=f"Linear V(s; theta)\n({vfa.n_features} parameters)",
                       ax=axes[1], show_values=False)

# Comparison: parameter count
methods = ['Tabular', 'Linear VFA', 'Neural Net\n(2 layers)']
params_needed = [state_size * action_size, vfa.n_features, 128 + 64 + action_size]
generalization = [0, 0.5, 0.9]  # Ability to generalize to unseen states

x = np.arange(len(methods))
width = 0.35
axes[2].bar(x - width/2, params_needed, width, label='Parameters',
            color=COLORS['rl'], alpha=0.85)
ax2 = axes[2].twinx()
ax2.bar(x + width/2, generalization, width, label='Generalization',
        color=COLORS['epistemic'], alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods)
axes[2].set_ylabel("Number of Parameters", color=COLORS['rl'])
ax2.set_ylabel("Generalization Ability", color=COLORS['epistemic'])
axes[2].set_title("Parameters vs Generalization")
axes[2].legend(loc='upper left', fontsize=9)
ax2.legend(loc='upper right', fontsize=9)

plt.suptitle("Tabular vs Function Approximation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nTabular: {state_size * action_size} parameters (one per state-action)")
print(f"Linear VFA: {vfa.n_features} parameters (generalizes across states!)")
print("Neural net: ~200 parameters, but can learn complex nonlinear features")

## 7.3 DQN: The Deep Learning Breakthrough

DQN (Mnih et al., 2015) combined Q-learning with deep neural networks. Two key innovations made it stable:

### Experience Replay
Store transitions $(s, a, r, s')$ in a replay buffer. Sample random mini-batches for training. This:
- Breaks correlations between consecutive samples
- Reuses data efficiently (each transition trained on many times)
- Like Dyna's model-based replay, but without learning a model!

### Target Networks
Use a separate "target" network $\hat{Q}(s',a'; \theta^-)$ for computing TD targets. Update $\theta^-$ slowly. This:
- Prevents the moving target problem (chasing your own tail)
- Stabilizes learning dramatically

### The DQN Update

$$L(\theta) = \mathbb{E}\left[\left(r + \gamma \max_{a'} \hat{Q}(s', a'; \theta^-) - Q(s, a; \theta)\right)^2\right]$$

```
for each training step:
    Sample mini-batch (s, a, r, s', done) from replay buffer
    Compute target: y = r + gamma * max_a' Q_target(s', a')
    Minimize MSE: L = (y - Q(s, a))^2
    Periodically: Q_target <- Q  (copy weights)
```

In [ ]:
# Illustrate experience replay: why random sampling helps

# Simulate a trajectory through a grid
n_steps = 100
grid_size_demo = 10

# Sequential experience (correlated)
sequential_states = []
pos = [0, 0]
for _ in range(n_steps):
    move = np.random.randint(4)
    if move == 0 and pos[0] > 0: pos[0] -= 1
    elif move == 1 and pos[0] < grid_size_demo-1: pos[0] += 1
    elif move == 2 and pos[1] > 0: pos[1] -= 1
    elif move == 3 and pos[1] < grid_size_demo-1: pos[1] += 1
    sequential_states.append(pos[0] * grid_size_demo + pos[1])

# Random replay (decorrelated)
replay_buffer = list(range(grid_size_demo**2)) * 2  # States seen at various times
replay_sample = np.random.choice(replay_buffer, size=n_steps, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Sequential states over time
axes[0].plot(sequential_states, 'o-', color=COLORS['danger'], markersize=3,
             alpha=0.7, linewidth=1)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("State Index")
axes[0].set_title("Sequential Experience\n(highly correlated)")
axes[0].grid(True, alpha=0.3)

# Replay buffer samples
axes[1].plot(replay_sample, 'o', color=COLORS['success'], markersize=3, alpha=0.7)
axes[1].set_xlabel("Step")
axes[1].set_ylabel("State Index")
axes[1].set_title("Experience Replay\n(decorrelated)")
axes[1].grid(True, alpha=0.3)

# Autocorrelation comparison
from numpy.fft import fft, ifft
def autocorr(x, max_lag=30):
    x = x - np.mean(x)
    result = np.correlate(x, x, mode='full')
    result = result[len(result)//2:]
    return result[:max_lag] / result[0] if result[0] > 0 else result[:max_lag]

ac_seq = autocorr(np.array(sequential_states, dtype=float))
ac_replay = autocorr(np.array(replay_sample, dtype=float))

axes[2].plot(ac_seq, color=COLORS['danger'], linewidth=2, label='Sequential')
axes[2].plot(ac_replay, color=COLORS['success'], linewidth=2, label='Replay')
axes[2].axhline(y=0, color='black', linewidth=0.5)
axes[2].set_xlabel("Lag")
axes[2].set_ylabel("Autocorrelation")
axes[2].set_title("Temporal Correlation")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle("Experience Replay Breaks Temporal Correlations",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Sequential data has high autocorrelation -> biased gradient estimates.")
print("Replay buffer samples are nearly uncorrelated -> stable learning.")

## 7.4 PPO and SAC: Modern Workhorses

DQN was the beginning, but modern deep RL has moved to more stable and sample-efficient algorithms.

### Proximal Policy Optimization (PPO) — Schulman et al., 2017

PPO is an actor-critic method with a **clipped surrogate objective** that prevents destructively large policy updates:

$$L^{\text{CLIP}} = \mathbb{E}\left[\min\left(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t\right)\right]$$

where $r_t(\theta) = \pi_\theta(a_t|s_t) / \pi_{\theta_{\text{old}}}(a_t|s_t)$ is the probability ratio.

### Soft Actor-Critic (SAC) — Haarnoja et al., 2018

SAC maximizes both reward AND entropy:

$$J(\pi) = \sum_t \mathbb{E}\left[r(s_t, a_t) + \alpha H(\pi(\cdot|s_t))\right]$$

This **maximum entropy RL** objective is the closest RL gets to free energy minimization! SAC is the RL algorithm that most directly corresponds to Active Inference.

> See the [neuro-nav deep RL tutorial](https://github.com/awjuliani/neuro-nav/blob/main/tutorials/deeprl_tutorial.ipynb) for hands-on training with PPO and SAC agents on neuro-nav environments.

In [ ]:
# Overview of deep RL algorithm landscape
# (This cell is conceptual/visual rather than training deep networks)

fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Algorithm positions in sample-efficiency vs stability space
algorithms = {
    'REINFORCE': (0.2, 0.3),
    'DQN': (0.5, 0.6),
    'A2C': (0.45, 0.5),
    'PPO': (0.6, 0.85),
    'SAC': (0.8, 0.9),
    'TD3': (0.75, 0.8),
    'Dreamer\n(model-based)': (0.9, 0.7),
}

colors_alg = {
    'REINFORCE': COLORS['neutral'],
    'DQN': COLORS['rl'],
    'A2C': '#66BB6A',
    'PPO': '#1565C0',
    'SAC': '#7B1FA2',
    'TD3': '#00897B',
    'Dreamer\n(model-based)': COLORS['aif'],
}

for name, (x, y) in algorithms.items():
    ax.scatter(x, y, s=200, color=colors_alg[name], zorder=5, edgecolors='black')
    ax.annotate(name, (x, y), textcoords="offset points",
                xytext=(12, 8), fontsize=10, fontweight='bold')

# Add AIF region
from matplotlib.patches import FancyBboxPatch
rect = FancyBboxPatch((0.65, 0.55), 0.3, 0.4, boxstyle="round,pad=0.05",
                       facecolor=COLORS['aif'], alpha=0.1,
                       edgecolor=COLORS['aif'], linewidth=2)
ax.add_patch(rect)
ax.text(0.8, 0.58, "Active Inference\nregion", ha='center', fontsize=10,
        color=COLORS['aif'], fontstyle='italic')

ax.set_xlabel("Sample Efficiency", fontsize=12)
ax.set_ylabel("Training Stability", fontsize=12)
ax.set_title("Deep RL Algorithm Landscape", fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.set_ylim(0.15, 1.05)
ax.grid(True, alpha=0.3)

# Add arrow showing progress
ax.annotate('', xy=(0.85, 0.9), xytext=(0.25, 0.35),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2, ls='--'))
ax.text(0.5, 0.3, "Historical\nprogress", ha='center', fontsize=9,
        color='gray', fontstyle='italic', rotation=35)

plt.tight_layout()
plt.show()

print("SAC's maximum entropy objective is closest to Active Inference.")
print("Model-based methods (Dreamer) are the most sample efficient.")
print("AIF naturally occupies the high-efficiency, model-based region.")

## 7.5 Representation Learning: What Do Agents Learn?

Perhaps the most interesting aspect of deep RL is **what representations emerge**. The hidden layers of a deep RL agent learn features that are useful for decision-making.

We can study this even in tabular settings by looking at:
- **Q-table structure**: Which states have similar Q-values? These form natural "clusters"
- **Successor Representation** (Dayan, 1993): $M(s,s') = \mathbb{E}[\sum_t \gamma^t \mathbb{1}[s_t = s'] | s_0 = s]$

The SR is particularly interesting because it separates *what states lead to what states* (dynamics) from *what states are rewarding* (preferences). This is exactly the decomposition in Active Inference: the B-matrix captures dynamics, while C captures preferences.

In [ ]:
# Compare tabular Q-values and SR representations
np.random.seed(42)

env = GridEnv(template=GridTemplate.four_rooms,
              obs_type=GridObservation.index,
              size=GridSize.small)
state_size = env.grid_size ** 2
action_size = env.action_space.n

# Train TDQ and TDSR agents
tdq = TDQ(state_size, action_size, lr=0.1, gamma=0.99,
          poltype="softmax", beta=5.0)
tdsr = TDSR(state_size, action_size, lr=0.1, gamma=0.99,
            poltype="softmax", beta=5.0)

for ep in range(300):
    obs = env.reset()
    for step in range(500):
        action_q = tdq.sample_action(obs)
        action_sr = tdsr.sample_action(obs)
        # Use same action for both
        next_obs, reward, done, info = env.step(action_q)
        tdq.update((obs, action_q, next_obs, reward, done))
        tdsr.update((obs, action_sr, next_obs, reward, done))
        obs = next_obs
        if done:
            break

print("Training complete.")

In [ ]:
# Visualize: Q-table vs SR matrix
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

grid_side = int(np.sqrt(state_size))

# TDQ value function
v_tdq = np.max(tdq.q, axis=1)
if grid_side**2 == state_size:
    plot_value_heatmap(v_tdq, grid_side,
                       title="TDQ: V(s) = max_a Q(s,a)\n(learned values)",
                       ax=axes[0], show_values=False)

# TDSR value function
v_tdsr = np.max(tdsr.q, axis=1) if hasattr(tdsr, 'q') else np.zeros(state_size)
if grid_side**2 == state_size:
    plot_value_heatmap(v_tdsr, grid_side,
                       title="TDSR: V(s) = max_a Q_SR(s,a)\n(SR-derived values)",
                       ax=axes[1], show_values=False)

# SR matrix (shows state-state relationships)
if hasattr(tdsr, 'M'):
    sr_matrix = tdsr.M
elif hasattr(tdsr, 'sr'):
    sr_matrix = tdsr.sr
else:
    # Fallback: compute from Q and w if available
    sr_matrix = np.eye(state_size)

# Show a subset of the SR matrix
n_show = min(20, state_size)
im = axes[2].imshow(sr_matrix[:n_show, :n_show], cmap='Purples',
                    interpolation='nearest')
axes[2].set_title(f"SR Matrix M(s,s')\n(first {n_show} states)")
axes[2].set_xlabel("Future State s'")
axes[2].set_ylabel("Current State s")
plt.colorbar(im, ax=axes[2], shrink=0.8)

plt.suptitle("Representations: Values vs Successor Representation",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTDQ learns VALUES directly: V(s) = expected cumulative reward.")
print("TDSR learns the STRUCTURE of the environment (which states lead to which),")
print("then computes value as V(s) = M(s,:) . w (SR times reward weights).")
print("\nThe SR separates dynamics from reward — exactly like AIF's B and C matrices!")

In [ ]:
# PCA / dimensionality reduction of learned representations
from sklearn.decomposition import PCA

# Get representations: Q-values as features
# Q shape is (action_size, state_size), transpose to (state_size, action_size) for PCA
q_features = tdq.Q.T  # shape: (state_size, action_size)

# PCA to 2D
pca = PCA(n_components=2)
q_2d = pca.fit_transform(q_features)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Color by position in grid
if grid_side**2 == state_size:
    rows = np.arange(state_size) // grid_side
    cols = np.arange(state_size) % grid_side
    distances = np.sqrt((rows - grid_side + 1)**2 + (cols - grid_side + 1)**2)
else:
    distances = np.arange(state_size)

scatter = axes[0].scatter(q_2d[:, 0], q_2d[:, 1], c=distances,
                          cmap='viridis', s=30, alpha=0.8, edgecolors='gray',
                          linewidth=0.5)
plt.colorbar(scatter, ax=axes[0], label='Distance to goal')
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
axes[0].set_title("PCA of Q-value Representations\n(colored by distance to goal)")
axes[0].grid(True, alpha=0.3)

# Color by value
scatter2 = axes[1].scatter(q_2d[:, 0], q_2d[:, 1], c=v_tdq,
                           cmap='RdYlGn', s=30, alpha=0.8, edgecolors='gray',
                           linewidth=0.5)
plt.colorbar(scatter2, ax=axes[1], label='V(s)')
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
axes[1].set_title("PCA of Q-value Representations\n(colored by value)")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Learned Representations Capture Environment Structure",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nPCA variance explained: {pca.explained_variance_ratio_.sum():.1%}")
print("States near the goal cluster together in representation space.")
print("The Q-values implicitly learn a low-dimensional representation of the task.")

In [ ]:
# Dual Perspective Box
dual_perspective_box(
    rl_text=(
        "A deep RL agent's hidden layers learn <b>features phi(s)</b> that are "
        "useful for predicting reward. These features emerge implicitly through "
        "the reward gradient: the network learns what to represent because it's "
        "useful for maximizing return. The learned representation approximates "
        "the <b>sufficient statistics</b> of the task."
    ),
    aif_text=(
        "An AIF agent's generative model <b>P(o|s)</b> explicitly defines the "
        "mapping from hidden states to observations. Where RL learns phi(s) "
        "implicitly through reward, AIF specifies the observation model "
        "explicitly. <b>Deep AIF</b> (Module 12) bridges this gap by "
        "parameterizing P(o|s) with neural networks, learning both "
        "representations and the generative model."
    ),
    title="Implicit vs Explicit Representations"
)

## Key Takeaways

1. **Tabular methods don't scale**: state space grows exponentially with problem dimensions
2. **Function approximation** $V(s;\theta)$ enables generalization across similar states
3. **DQN** combined Q-learning with neural networks using experience replay and target networks
4. **SAC** (maximum entropy RL) is the closest RL algorithm to Active Inference
5. **Deep RL agents implicitly learn representations**; AIF agents explicitly specify generative models. Deep AIF (Module 12) unifies both.

---

## 7.7 Beyond Vanilla RNNs: The Modern Landscape

The story of training neural networks on cognitive tasks has a cautionary tale. **PsychRNN** ([Ehrlich et al., eNeuro 2021](https://github.com/murraylab/PsychRNN)) from the Murray Lab at Yale pioneered the "train RNNs on cognitive tasks" paradigm, offering biological constraints (Dale's law, E/I balance) in a convenient package. But it was built on TensorFlow 1.x — a framework that is now effectively dead. TF 1.x requires Python ≤3.8, targets obsolete CUDA versions, has no Apple Silicon builds, and uses session-based graph execution. PsychRNN is essentially uninstallable today.

The field has since fragmented into better tools:

| Project | Stack | Key Advance |
|---------|-------|-------------|
| [**NeuroGym**](https://github.com/neurogym/neurogym) (314 stars, active) | Framework-agnostic (Gymnasium API) | Decouples task definition from training — works with PyTorch, JAX, anything |
| [**NN4N**](https://github.com/nn4neurosim/nn4n) (NeurIPS 2024) | PyTorch | Multi-area CTRNNs, EIRNN, fine-grained connectivity control |
| [**MotorNet**](https://github.com/OlivierCod662/MotorNet) (eLife 2024) | PyTorch | Differentiable biomechanical effectors for motor control |
| [**Disentangled RNNs**](https://arxiv.org/abs/2305.16865) (DeepMind, NeurIPS 2023) | JAX/Haiku | Interpretable RNNs for cognitive model discovery |
| [**Jaxley**](https://github.com/jaxleyverse/jaxley) (Nature Methods 2025) | JAX | Differentiable biophysical neuron models at scale |
| [**SOFO**](https://arxiv.org/abs/2402.11867) (NeurIPS 2024) | JAX+PyTorch | Second-order optimization with constant memory — no BPTT needed |

But the bigger story is that **entirely new architectures** now outperform vanilla RNNs on cognitive tasks:

### State Space Models
[BrainMamba](https://arxiv.org/abs/2403.01190) (CHIL 2024) and NeuroMamba (NeurIPS 2025) use S4/Mamba architectures for brain activity encoding, outperforming prior methods on neural decoding with linear complexity in sequence length.

### Transformers as Cognitive Models
Whittington et al. (arXiv 2024) trained attention-only transformers on a human working memory task. The attention heads spontaneously specialized into functional roles mirroring frontostriatal circuits: key composition = input gating, query composition = output gating, with 70.2% attention focused on the stored register matching the target. No built-in gating was provided; it emerged purely from task optimization.

### Continuous-Time Networks
[Closed-form Continuous-time Networks (CfCs)](https://www.nature.com/articles/s42256-022-00556-7) (Hasani et al., Nature Machine Intelligence 2022) — inspired by biological neural dynamics with liquid time-constants. 100x faster than neural ODEs while matching or beating RNN/LSTM/GRU.

### Tiny Interpretable RNNs
Ji-An, Benna & Mattar ([Nature 2025](https://www.nature.com/articles/s41586-024-08145-x)) showed that RNNs with just **1–4 units**, analyzed as dynamical systems, outperform 30+ classical cognitive models on reward-learning tasks (reversal learning, drifting bandits, two-stage tasks). The low dimensionality enables exhaustive phase portrait analysis, revealing novel cognitive strategies invisible to standard model-fitting. See also the rebuttal by Katahira (bioRxiv Oct 2025) on interpretability risks.

### Multiplicative Couplings
bioRxiv July 2025: Inspired by thalamocortical interactions, multiplicative (not additive) interactions between feedforward and recurrent pathways enable Hebbian-weight amplification for context-dependent gating, dramatically improving learning speed and generalization on working memory and attention tasks.

### Symbolic Mechanisms Inside LLMs
Yang, Campbell, Webb & Cohen ([ICML 2025](https://arxiv.org/abs/2502.01235)) identified a three-stage symbolic architecture emerging inside trained LLMs: symbol abstraction heads → symbolic induction heads → retrieval heads. Structured symbolic computation emerges within neural networks when tasks demand it.

### Continuous Thought Machines
Sakana AI (May 2025) introduced an architecture where each neuron processes a temporal history of incoming signals, with information encoded in synchronization patterns of neural activity — inspired by neural synchrony theories. The model naturally adapts its processing time to input complexity (harder inputs = more internal "ticks"), analogous to human reaction-time effects.

### Open Research Gaps

These opportunities remain wide open and are directly addressable with the tools in this curriculum:

1. No systematic **SSM vs. RNN vs. transformer comparison** on the same classical cognitive tasks, with internal representations compared against neural recordings
2. No **continuous-time models** (CfCs, LTCs) trained on standard cognitive tasks — surprising given biological dynamics are continuous
3. No **differentiable active inference agents** on the cognitive neuroscience task battery (e.g., [NeuroGym](https://github.com/neurogym/neurogym))
4. No **Koopman spectral analysis** ([Ostrow et al., NeurIPS 2023](https://arxiv.org/abs/2301.10197) — Dynamical Similarity Analysis) applied to networks trained on parametric working memory or context-dependent integration

---

## Further Reading

- Mnih, V. et al. (2015). Human-level control through deep reinforcement learning. *Nature*, 518. — The DQN paper.
- Schulman, J. et al. (2017). Proximal policy optimization algorithms. *arXiv*. — PPO.
- Haarnoja, T. et al. (2018). Soft Actor-Critic. *ICML*. — Maximum entropy RL, the bridge to AIF.
- Dayan, P. (1993). Improving generalization for temporal difference learning. *Neural Computation*. — The Successor Representation.
- Tschantz, A. et al. (2020). Reinforcement learning through active inference. *arXiv*. — Formal proof connecting deep RL and AIF.
- Ehrlich, D.B. et al. (2021). [PsychRNN](https://github.com/murraylab/PsychRNN). *eNeuro*. — Historical; TF 1.x, now superseded by the tools above.
- Hasani, R. et al. (2022). Closed-form continuous-depth models. *Nature Machine Intelligence*. — CfCs: 100x faster than neural ODEs.
- Ji-An, Li, Benna & Mattar (2025). [Tiny RNNs as cognitive models](https://www.nature.com/articles/s41586-024-08145-x). *Nature*. — 1–4 unit RNNs outperform 30+ classical models.
- Ostrow, M. et al. (2023). [Dynamical Similarity Analysis](https://arxiv.org/abs/2301.10197). *NeurIPS*. — Koopman-based comparison of trained network dynamics.
- Yang, Z. et al. (2025). [Symbolic mechanisms in LLMs](https://arxiv.org/abs/2502.01235). *ICML*. — Emergent symbolic computation in neural networks.

---

*Next: Module 8 — Generative Models and the Free Energy Principle (the transition to Active Inference!)*